# Preisanalyse von Gebrauchtwagen in Deutschland (AutoScout24, 2023)

## 1. Einleitung

Dieser Datensatz stammt von Kaggle und ist die Grundlage dieser Analyse.
Der Datensatz "Germany Used Cars Dataset 2023" ist von der Plattform Autoscout24 und hat über 200.000 Inserate (Beobachtungen).

(Quelle: wspirat, 2023, Kaggle, CC0)
https://www.kaggle.com/datasets/wspirat/germany-used-cars-dataset-2023/data

Ziel der Analyse ist es herauszufinden, welche Merkmale den Preis eines Gebrauchtwagens
bestimmen und wie gut sich der Preis vorhersagen lässt.
Diese Frage ist für einen Gebrauchtwagenhändler relevant, da er seine Fahrzeuge
marktgerecht bepreisen und günstige Einkaufsmöglichkeiten erkennen möchte.

Daraus ergeben sich zwei Forschungsfragen:

1. Welche Merkmale beeinflussen den Preis am stärksten?
2. Wie gut lässt sich der Preis anhand der Merkmale vorhersagen?

Als Zielvariable dient der Angebotspreis (`price_in_euro`).

## 2. Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Darstellung
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# relative Pfade (reproduzierbar)
DATA_RAW = Path("../data/raw/data.csv")
DATA_PROCESSED = Path("../data/processed/data_processed.csv")
IMG_DIR = Path("../images")
IMG_DIR.mkdir(exist_ok=True)

# Daten laden
df = pd.read_csv(DATA_RAW)
print("Shape:", df.shape)
df.head(20)

Shape: (251079, 15)


,Unnamed: 0,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description
0,0,alfa-romeo,Alfa Romeo GTV,red,10/1995,1995,1300,148,201,Manual,Petrol,"10,9 l/100 km",260 g/km,160500.0,2.0 V6 TB
1,1,alfa-romeo,Alfa Romeo 164,black,02/1995,1995,24900,191,260,Manual,Petrol,NaN,- (g/km),190000.0,"Q4 Allrad, 3.2L GTA"
2,2,alfa-romeo,Alfa Romeo Spider,black,02/1995,1995,5900,110,150,Unknown,Petrol,NaN,- (g/km),129000.0,ALFA ROME 916
3,3,alfa-romeo,Alfa Romeo Spider,black,07/1995,1995,4900,110,150,Manual,Petrol,"9,5 l/100 km",225 g/km,189500.0,2.0 16V Twin Spark L
4,4,alfa-romeo,Alfa Romeo 164,red,11/1996,1996,17950,132,179,Manual,Petrol,"7,2 l/100 km",- (g/km),96127.0,"3.0i Super V6, absoluter Topzustand !"
5,5,alfa-romeo,Alfa Romeo Spider,red,04/1996,1996,7900,110,150,Manual,Petrol,"9,5 l/100 km",225 g/km,47307.0,2.0 16V Twin Spark
6,6,alfa-romeo,Alfa Romeo 145,red,12/1996,1996,3500,110,150,Manual,Petrol,"8,8 l/100 km",210 g/km,230000.0,Quadrifoglio
7,7,alfa-romeo,Alfa Romeo 164,black,07/1996,1996,5500,132,179,Manual,Petrol,"13,4 l/100 km",320 g/km,168000.0,(3.0) V6 Super
8,8,alfa-romeo,Alfa Romeo Spider,black,07/1996,1996,8990,141,192,Manual,Petrol,11 l/100 km,265 g/km,168600.0,|HU:neu|Klimaanlage|Youngtimer|
9,9,alfa-romeo,Alfa Romeo Spider,black,01/1996,1996,6976,110,150,Manual,Petrol,"9,2 l/100 km",220 g/km,99000.0,2.0 T.Spark L *Klima *2.Hand *Zahnriemen


## 3. Datenqualität

### 3.1 Struktur & Datentypen

In [2]:
# Überblick: Spalten, Datentypen (Dtype), vorhandene Werte
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251079 entries, 0 to 251078
Data columns (total 15 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Unnamed: 0                251079 non-null  int64  
 1   brand                     251079 non-null  object 
 2   model                     251079 non-null  object 
 3   color                     250913 non-null  object 
 4   registration_date         251075 non-null  object 
 5   year                      251079 non-null  object 
 6   price_in_euro             251079 non-null  object 
 7   power_kw                  250945 non-null  object 
 8   power_ps                  250950 non-null  object 
 9   transmission_type         251079 non-null  object 
 10  fuel_type                 251079 non-null  object 
 11  fuel_consumption_l_100km  224206 non-null  object 
 12  fuel_consumption_g_km     251079 non-null  object 
 13  mileage_in_km             250927 non-null  f

`df.info()` zeigt, dass 13 der 15 Spalten als Text (`object`) gespeichert sind,
obwohl mehrere davon Zahlen enthalten. `year`, `power_kw` und `power_ps` sollten
Ganzzahlen (int) sein, `price_in_euro` und `fuel_consumption_l_100km` Kommazahlen
(float) und `registration_date` ein Datum. Die Spalte `Unnamed: 0` ist lediglich
ein überflüssiger Index. Die falschen Datentypen deuten auf verunreinigte Zellen
in einzelnen Zeilen hin; die Umwandlung erfolgt im späteren
Bereinigungsschritt. Zudem ist der Non-Null-Count nicht überall gleich: Mehrere Spalten enthalten
fehlende Werte, am deutlichsten `fuel_consumption_l_100km` mit nur 224.206 von
251.079 Einträgen. Diese werden in Kapitel 3.2 genauer untersucht.

### 3.2 Fehlende Werte

In [3]:
# Anteil echter fehlender Werte (NaN) je Spalte in %, absteigend sortiert
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

fuel_consumption_l_100km    10.70
color                        0.07
mileage_in_km                0.06
power_ps                     0.05
power_kw                     0.05
brand                        0.00
Unnamed: 0                   0.00
price_in_euro                0.00
year                         0.00
registration_date            0.00
model                        0.00
fuel_type                    0.00
transmission_type            0.00
fuel_consumption_g_km        0.00
offer_description            0.00
dtype: float64

In [4]:
# Versteckte Missings prüfen – Platzhalter statt NaN
df["fuel_consumption_g_km"].value_counts().head()

fuel_consumption_g_km
- (g/km)    35809
0 g/km       8533
119 g/km     4813
114 g/km     3882
139 g/km     3389
Name: count, dtype: int64

In [5]:
df["transmission_type"].value_counts()

transmission_type
Automatic         131749
Manual            117869
Unknown             1144
Semi-automatic       317
Name: count, dtype: int64

In [6]:
# "0 g/km" plausibel nur bei Elektrofahrzeugen
mask = df["fuel_consumption_g_km"].str.strip() == "0 g/km"
df.loc[mask, "fuel_type"].value_counts()

fuel_type
Petrol           3472
Electric         2930
Diesel           1706
Hybrid            266
LPG                62
Hydrogen           57
CNG                15
Other              12
Diesel Hybrid      12
Ethanol             1
Name: count, dtype: int64

In [7]:
# "0 g/km" ist nur bei Nullemissions-Antrieben plausibel (Elektro, Wasserstoff)
mask_0 = df["fuel_consumption_g_km"].str.strip() == "0 g/km"
nullemission = ["Electric", "Hydrogen"]
implausibel = (mask_0 & ~df["fuel_type"].isin(nullemission)).sum()
print("Implausible 0 g/km (Verbrenner):", implausibel)

Implausible 0 g/km (Verbrenner): 5546


Nur `fuel_consumption_l_100km` weist mit 10,7 % viele **echte** fehlende Werte (NaN) auf.
Zusätzlich existieren **versteckte** fehlende Werte, die `isna()` nicht erkennt.
`fuel_consumption_g_km` enthält 35.809-mal den Platzhalter `"- (g/km)"`.
In `transmission_type` steht 1.144-mal `"Unknown"`.
Die tatsächliche Fehlerquote ist damit höher, als der reine NaN-Check vermuten lässt.
Darüber hinaus enthält `fuel_consumption_g_km` 8.533-mal den Wert `0 g/km`.
Der Kreuzvergleich mit `fuel_type` zeigt, dass dies nur bei Elektrofahrzeugen
(2.930) und Wasserstofffahrzeugen (57) plausibel ist, während er bei Verbrennern (5.546) fehlerhaft ist.

Die Platzhalter werden im Bereinigungsschritt in echte NaN
umgewandelt. Aufgrund der hohen Fehlerquote und der Redundanz beider
Verbrauchsspalten werden diese als Prädiktoren kritisch geprüft.

### 3.3 Duplikate

In [8]:
# Naiver Check – zählt volle Zeilen inkl. künstlichem Index
df.duplicated().sum()

np.int64(0)

In [9]:
# Echte Duplikate: Anzahl und Anteil
dupes = df.drop(columns="Unnamed: 0").duplicated().sum()
print(f"Duplikate: {dupes} ({dupes / len(df) * 100:.2f} %)")

Duplikate: 6353 (2.53 %)


Der naive Duplikat-Check liefert 0, weil die künstliche Index-Spalte
`Unnamed: 0` jede Zeile eindeutig macht und echte Duplikate verdeckt.
Schließt man diese Spalte aus, zeigen sich 6.353 doppelte Inserate (2.53 %).

Die Duplikate werden im Bereinigungsschritt entfernt.

## 4. Datenbereinigung

### 4.1 Kopie erstellen, Index & Duplikate entfernen

In [10]:
# Rohdatenkopie, (df) bleiben unverändert
df_clean = df.copy()

# Überflüssige Index-Spalte entfernen
df_clean = df_clean.drop(columns="Unnamed: 0")

# Echte Duplikate entfernen
vorher = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Entfernte Duplikate: {vorher - len(df_clean)} | Verbleibend: {len(df_clean)}")

Entfernte Duplikate: 6353 | Verbleibend: 244726


Als Erstes wird eine Kopie erstellt, um die Rohdaten unverändert zu
lassen. Die überflüssige Index-Spalte `Unnamed: 0` wird entfernt und die
6.353 echten Duplikate gelöscht und es verbleiben 244.726 Inserate.

### 4.2 Datentypen korrigieren

In [11]:
# Preis als Zahl (float); errors="coerce" -> ungültiger Text wird NaN
df_clean["price_in_euro"] = pd.to_numeric(df_clean["price_in_euro"], errors="coerce")

# Ganzzahl-Merkmale als Int64 (Ganzzahl, verträgt aber NaN)
for col in ["year", "power_kw", "power_ps"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

In [12]:
# Zulassungsdatum (Format MM/JJJJ) in echtes Datum umwandeln
df_clean["registration_date"] = pd.to_datetime(
    df_clean["registration_date"], format="%m/%Y", errors="coerce"
)

In [13]:
# Prüfen: Typen korrekt? Wie viele NaN sind durch die Umwandlung entstanden?
print(df_clean.dtypes)
print("\nNeue NaN je Spalte:")
print(df_clean.isna().sum())

brand                               object
model                               object
color                               object
registration_date           datetime64[ns]
year                                 Int64
price_in_euro                      float64
power_kw                             Int64
power_ps                             Int64
transmission_type                   object
fuel_type                           object
fuel_consumption_l_100km            object
fuel_consumption_g_km               object
mileage_in_km                      float64
offer_description                   object
dtype: object

Neue NaN je Spalte:
brand                           0
model                           0
color                          84
registration_date             184
year                          180
price_in_euro                 184
power_kw                      277
power_ps                      207
transmission_type               0
fuel_type                       0
fuel_consumption_l_100k

Nach der Umwandlung liegen alle Spalten im korrekten Typ vor (`year`, `power_kw`,
`power_ps` als Int64, `price_in_euro`/`mileage_in_km` als float und `registration_date`
als Datum). Durch `errors="coerce"` wurden die verschobenen Zeilen als NaN
markiert (z. B. 180 in `year`, 277 in `power_kw`). Diese werden in 4.3 behandelt.

### 4.3 Platzhalter & unplausible Werte behandeln

In [14]:
# Versteckte Platzhalter in echte NaN umwandeln
df_clean["transmission_type"] = df_clean["transmission_type"].replace("Unknown", np.nan)
df_clean["fuel_type"] = df_clean["fuel_type"].replace("Unknown", np.nan)

In [15]:
# Wertebereich von year prüfen
print(df_clean["year"].min(), df_clean["year"].max())

1995 27449


In [16]:
# Wie viele Fahrzeuge haben ein unplausibles Baujahr (> 2023)?
print("Unplausible Jahre (>2023):", (df_clean["year"] > 2023).sum())

# Unplausible Jahre auf NaN setzen (Datensatz deckt nur 1995–2023 ab)
df_clean.loc[df_clean["year"] > 2023, "year"] = pd.NA

Unplausible Jahre (>2023): 4


Die Platzhalter `"Unknown"` in `transmission_type` und `fuel_type` wurden in
echte NaN umgewandelt. Zudem wiesen 4 Fahrzeuge ein unplausibles Baujahr über
2023 auf (Datensatz deckt 1995–2023 ab); diese wurden auf NaN gesetzt.

### 4.4 Finale Spaltenauswahl & Export

In [17]:
# Kontrolle: Struktur des bereinigten Datensatzes
df_clean.head(20)

,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description
0,alfa-romeo,Alfa Romeo GTV,red,1995-10-01,1995,1300.0,148,201,Manual,Petrol,"10,9 l/100 km",260 g/km,160500.0,2.0 V6 TB
1,alfa-romeo,Alfa Romeo 164,black,1995-02-01,1995,24900.0,191,260,Manual,Petrol,NaN,- (g/km),190000.0,"Q4 Allrad, 3.2L GTA"
2,alfa-romeo,Alfa Romeo Spider,black,1995-02-01,1995,5900.0,110,150,NaN,Petrol,NaN,- (g/km),129000.0,ALFA ROME 916
3,alfa-romeo,Alfa Romeo Spider,black,1995-07-01,1995,4900.0,110,150,Manual,Petrol,"9,5 l/100 km",225 g/km,189500.0,2.0 16V Twin Spark L
4,alfa-romeo,Alfa Romeo 164,red,1996-11-01,1996,17950.0,132,179,Manual,Petrol,"7,2 l/100 km",- (g/km),96127.0,"3.0i Super V6, absoluter Topzustand !"
5,alfa-romeo,Alfa Romeo Spider,red,1996-04-01,1996,7900.0,110,150,Manual,Petrol,"9,5 l/100 km",225 g/km,47307.0,2.0 16V Twin Spark
6,alfa-romeo,Alfa Romeo 145,red,1996-12-01,1996,3500.0,110,150,Manual,Petrol,"8,8 l/100 km",210 g/km,230000.0,Quadrifoglio
7,alfa-romeo,Alfa Romeo 164,black,1996-07-01,1996,5500.0,132,179,Manual,Petrol,"13,4 l/100 km",320 g/km,168000.0,(3.0) V6 Super
8,alfa-romeo,Alfa Romeo Spider,black,1996-07-01,1996,8990.0,141,192,Manual,Petrol,11 l/100 km,265 g/km,168600.0,|HU:neu|Klimaanlage|Youngtimer|
9,alfa-romeo,Alfa Romeo Spider,black,1996-01-01,1996,6976.0,110,150,Manual,Petrol,"9,2 l/100 km",220 g/km,99000.0,2.0 T.Spark L *Klima *2.Hand *Zahnriemen


In [18]:
# Prüfen: Ist das Jahr aus registration_date redundant zu year?
maske = df_clean["registration_date"].notna() & df_clean["year"].notna()
abweichungen = (df_clean.loc[maske, "registration_date"].dt.year
                != df_clean.loc[maske, "year"]).sum()
print("Verglichene Zeilen:", maske.sum(), "| Abweichende Jahre:", abweichungen)

Verglichene Zeilen: 244542 | Abweichende Jahre: 0


In [19]:
# Monat der Erstzulassung als eigenes numerisches Merkmal
df_clean["registration_month"] = df_clean["registration_date"].dt.month

# Kontrolle: Jahr + Monat sauber getrennt?
df_clean[["registration_date", "year", "registration_month"]].head()

,registration_date,year,registration_month
0,1995-10-01,1995,10.0
1,1995-02-01,1995,2.0
2,1995-02-01,1995,2.0
3,1995-07-01,1995,7.0
4,1996-11-01,1996,11.0


In [20]:
# Nicht benötigte Spalten entfernen (begründet)
df_clean = df_clean.drop(columns=[
    "registration_date",         # atomar zerlegt in year + registration_month
    "offer_description",         # Freitext, kein Prädiktor
    "fuel_consumption_l_100km",  # ~11% fehlend
    "fuel_consumption_g_km",     # redundant + fehlerhafte Werte
    "power_kw",                  # redundant zu power_ps
])

# Zeilen mit fehlenden Werten entfernen
vorher = len(df_clean)
df_clean = df_clean.dropna()
print(f"Entfernte Zeilen: {vorher - len(df_clean)} | Verbleibend: {len(df_clean)}")

# Ganzzahl-Spalten final auf int (jetzt ohne NaN)
for col in ["year", "registration_month", "power_ps"]:
    df_clean[col] = df_clean[col].astype(int)

# Export
df_clean.to_csv(DATA_PROCESSED, index=False)
print("Gespeichert:", DATA_PROCESSED, "| Shape:", df_clean.shape)

Entfernte Zeilen: 1666 | Verbleibend: 243060
Gespeichert: ..\data\processed\data_processed.csv | Shape: (243060, 10)


In [21]:
# Test: gespeicherte Datei zurücklesen und prüfen
df_check = pd.read_csv(DATA_PROCESSED)
print("Shape:", df_check.shape)
df_check.info()

Shape: (243060, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243060 entries, 0 to 243059
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   brand               243060 non-null  object 
 1   model               243060 non-null  object 
 2   color               243060 non-null  object 
 3   year                243060 non-null  int64  
 4   price_in_euro       243060 non-null  float64
 5   power_ps            243060 non-null  int64  
 6   transmission_type   243060 non-null  object 
 7   fuel_type           243060 non-null  object 
 8   mileage_in_km       243060 non-null  float64
 9   registration_month  243060 non-null  int64  
dtypes: float64(2), int64(3), object(5)
memory usage: 18.5+ MB


In der finalen Auswahl wurden Spalten entfernt, die entweder redundant sind
(`power_kw` doppelt zu `power_ps`, `registration_date` zu `year`) oder stark
unvollständig (die beiden Verbrauchsspalten). Aus der Erstzulassung wurde der
Monat als eigenes numerisches Merkmal extrahiert, sodass Jahr und Monat atomar
getrennt vorliegen. Übrig bleiben 243.060 Inserate mit 10 aussagekräftigen
Merkmalen ohne fehlende Werte. Der Datensatz wurde als `data_processed.csv`
gespeichert; ein erneutes Einlesen bestätigte den fehlerfreien Export als
Grundlage für EDA und Modellierung.

## 5. Explorative Datenanalyse

### 5.1 Zielvariable: Preis

### 5.2 Numerische Merkmale

### 5.3 Kategoriale Merkmale

### 5.4 Zusammenhänge & Korrelationen

## 6. Zwischenfazit